In [2]:
from pathlib import Path
import pandas as pd

# Paths
DATA_DIR = Path("/kaggle/input/heartbeat")
TRAIN_CSV = DATA_DIR / "mitbih_train.csv"
TEST_CSV  = DATA_DIR / "mitbih_test.csv"

LABEL_COL = 187  # dernière colonne = label


def load_csv(path: Path) -> pd.DataFrame:
    """Charge un CSV Kaggle MIT-BIH (sans header)."""
    if not path.exists():
        raise FileNotFoundError(f"Fichier introuvable : {path}")
    return pd.read_csv(path, header=None)


# Load data
train_df = load_csv(TRAIN_CSV)
test_df  = load_csv(TEST_CSV)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

train_df.head()

Train shape: (87554, 188)
Test shape : (21892, 188)


,0,1,2,3,4,5,6,7,8,9,...,178,179,180,181,182,183,184,185,186,187
0,0.977941,0.926471,0.681373,0.245098,0.154412,0.191176,0.151961,0.085784,0.058824,0.049020,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.960114,0.863248,0.461538,0.196581,0.094017,0.125356,0.099715,0.088319,0.074074,0.082621,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.000000,0.659459,0.186486,0.070270,0.070270,0.059459,0.056757,0.043243,0.054054,0.045946,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.925414,0.665746,0.541436,0.276243,0.196133,0.077348,0.071823,0.060773,0.066298,0.058011,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.967136,1.000000,0.830986,0.586854,0.356808,0.248826,0.145540,0.089202,0.117371,0.150235,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [3]:
import numpy as np

# Features = toutes les colonnes sauf la dernière, Label = dernière colonne
X_train = train_df.iloc[:, :LABEL_COL].to_numpy(dtype=np.float32)
y_train = train_df.iloc[:, LABEL_COL].to_numpy(dtype=np.int64)

X_test = test_df.iloc[:, :LABEL_COL].to_numpy(dtype=np.float32)
y_test = test_df.iloc[:, LABEL_COL].to_numpy(dtype=np.int64)

# Sanity checks
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_test : {X_test.shape} | y_test : {y_test.shape}")

print("Train classes:", np.unique(y_train))
print("Test classes :", np.unique(y_test))

# Quick value checks
print("X_train range:", float(X_train.min()), "->", float(X_train.max()))
print("Any NaN in X_train?", np.isnan(X_train).any())

X_train: (87554, 187) | y_train: (87554,)
X_test : (21892, 187) | y_test : (21892,)
Train classes: [0 1 2 3 4]
Test classes : [0 1 2 3 4]
X_train range: 0.0 -> 1.0
Any NaN in X_train? False


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Pipeline propre : scaling + modèle
model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=1000,
        n_jobs=-1,
        multi_class="auto"
    ))
])

# Entraînement
model.fit(X_train, y_train)

# Prédictions
y_pred = model.predict(X_test)

# Évaluation
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Test accuracy: 0.9147

Classification report:
              precision    recall  f1-score   support

           0       0.92      0.98      0.95     18118
           1       0.82      0.41      0.55       556
           2       0.66      0.33      0.44      1448
           3       0.59      0.37      0.45       162
           4       0.96      0.88      0.92      1608

    accuracy                           0.91     21892
   macro avg       0.79      0.60      0.66     21892
weighted avg       0.90      0.91      0.90     21892

